In [1]:
import requests

In [2]:
#URL del sitio que utilizaremos como origen de datos
url="https://books.toscrape.com/"

In [ ]:
#Realizamos una solicitud HTTP al servidor
#timeput es el tiempo máximo antes de lanzar un error si el servidor no responde
response = requests.get(url,timeout=10)

In [9]:
#Mostramos el código de respuesta
print("Código de respuesta:",response.status_code)

Código de respuesta: 200


In [ ]:
#Mostramos una parte del contenido recibido
#Lo da en un lenguaje html, da el creador, descripción y el titulo
#Es lo que vamos cuano en la pagina hacemos inspeccionar
print(response.text[:1000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="static/oscar/favicon.

In [ ]:
# Convertimos el HTML recibido en un objeto BeautifulSoup para poder analizarlo
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

#response.text: Toma el contenido HTML en texto plano que devolvió el servidor en la solicitud previa.

#"html.parser": Especifica el analizador sintáctico (parser) nativo de Python que usará la librería para interpretar la estructura del texto HTML.

#soup: Crea una estructura en árbol del documento HTML (un objeto BeautifulSoup). A partir de esta variable podrás buscar etiquetas, extraer textos, atributos, enlaces, tablas, etc., usando métodos como soup.find() o soup.find_all().


In [43]:
#Buscamos todos los elementos que representan un libro
#Todo lo que diga articulo
books = soup.find_all("article", class_="product_pod")

In [44]:
#Mostramos cuantos libros encontramos
print ("Libros encontrados:", len(books))

Libros encontrados: 20


In [45]:
#Seleccionamos el primer libro encontrado
book = books[0]

In [46]:
#Extraemos el título
title=book.h3.a["title"]

In [47]:
#Mostramos el título
print("Titulo:",title)

Titulo: A Light in the Attic


In [48]:
#Extraemos el precio y lo mostramos
price = book.find(
    "p",
    class_="price_color"
).get_text(strip=True)

#Mostramos el precio
print("Precio",price)

Precio Â£51.77


In [49]:
#Extraemos la disponibilidad
availability=book.find(
    "p",
    class_="instock availability"
).get_text(strip="True")

In [50]:
#Mostramos los resultados
print("Titulo",title)
print("Precio",price)
print("Disponibilidad:",availability)

Titulo A Light in the Attic
Precio Â£51.77
Disponibilidad: In stock


In [51]:
#Extraemos el elemento que contiene la calificación
rating_element = book.find(
    "p",
    class_="star-rating"
)

In [52]:
#Obtenemos la clase que representa la calificación
rating_text = rating_element["class"][1]
print("Calificacion:", rating_text)


Calificacion: Three


In [62]:
#Como la calificación viene en letras o texto necesitamos hacer como una limpieza de ponerlo en número para una escala
rating_map={
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
}

rating = rating_map.get(rating_text)

In [57]:
#Lista donde almacenaremos los reusltados
data = []

In [53]:
#Extraemos la URL
relative_url = book.h3.a["href"]

In [54]:
#Convertimos la URL relativa a una URL completa
from urllib.parse import urljoin
product_url = urljoin(url, relative_url)

In [55]:
#Mostramos los resultados
print("Titulo:",title)
print("Precio:", price)
print("Disponibilidad",availability)
print("Calificación:",rating)
print("URL:", product_url)

Titulo: A Light in the Attic
Precio: Â£51.77
Disponibilidad In stock
Calificación: 3
URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [ ]:
#Para guardar en un data frame
from urllib.parse import urljoin

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Luego la inicialización de la lista y el bucle
data = []

# 2. Recorremos cada libro (book) dentro de la lista de libros (books)
for book in books:
    # Extraemos el título
    title = book.h3.a["title"]

    # Extraemos el precio
    price = book.find("p", class_="price_color").get_text(strip=True)

    # Extraemos la disponibilidad (CORREGIDO: get_text(strip=True))
    availability = book.find("p", class_="instock availability").get_text(
        strip=True
    )

    # Extraemos la calificación / rating
    rating_element = book.find("p", class_="star-rating")
    rating_text = rating_element["class"][1]
    rating = rating_map.get(rating_text)

    # Extraemos la URL (AQUÍ ESTABA EL ERROR: AHORA ESTÁ DENTRO DEL FOR)
    relative_url = book.h3.a["href"]
    product_url = urljoin(url, relative_url)

    # 3. Guardamos los datos en la lista 'data' (DENTRO DEL FOR)
    data.append(
        {
            "titulo": title,
            "precio": price,
            "disponibilidad": availability,
            "rating": rating,
            "url": product_url,
        }
    )

# Mostramos los resultados (esto va FUERA del for)
print("Total de libros guardados:", len(data))
print(data)

Total de libros guardados: 20
[{'titulo': 'A Light in the Attic', 'precio': 'Â£51.77', 'disponibilidad': 'In stock', 'rating': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}, {'titulo': 'Tipping the Velvet', 'precio': 'Â£53.74', 'disponibilidad': 'In stock', 'rating': 1, 'url': 'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'}, {'titulo': 'Soumission', 'precio': 'Â£50.10', 'disponibilidad': 'In stock', 'rating': 1, 'url': 'https://books.toscrape.com/catalogue/soumission_998/index.html'}, {'titulo': 'Sharp Objects', 'precio': 'Â£47.82', 'disponibilidad': 'In stock', 'rating': 4, 'url': 'https://books.toscrape.com/catalogue/sharp-objects_997/index.html'}, {'titulo': 'Sapiens: A Brief History of Humankind', 'precio': 'Â£54.23', 'disponibilidad': 'In stock', 'rating': 5, 'url': 'https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html'}, {'titulo': 'The Requiem Red', 'precio': 'Â£22.65', 'dispon

In [69]:
#Mostramos los primeros tres libros
print("\nPrimeros libros extraidos:\n")

for libro in data[:3]:
    print(libro)
    print()


Primeros libros extraidos:

{'titulo': 'A Light in the Attic', 'precio': 'Â£51.77', 'disponibilidad': 'In stock', 'rating': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}

{'titulo': 'Tipping the Velvet', 'precio': 'Â£53.74', 'disponibilidad': 'In stock', 'rating': 1, 'url': 'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'}

{'titulo': 'Soumission', 'precio': 'Â£50.10', 'disponibilidad': 'In stock', 'rating': 1, 'url': 'https://books.toscrape.com/catalogue/soumission_998/index.html'}



In [ ]:
import pandas as pd

In [70]:
#Comprobamos cuantos registros guardamos
print("Total de registros guardados:", len(data))

Total de registros guardados: 20


In [71]:
#Convertimos los datos a un data frame
df=pd.DataFrame(data)

In [72]:
#Mostramos el data frame
print("\nDataFrame generado:\n")
print(df)


DataFrame generado:

                                               titulo   precio disponibilidad  \
0                                A Light in the Attic  Â£51.77       In stock   
1                                  Tipping the Velvet  Â£53.74       In stock   
2                                          Soumission  Â£50.10       In stock   
3                                       Sharp Objects  Â£47.82       In stock   
4               Sapiens: A Brief History of Humankind  Â£54.23       In stock   
5                                     The Requiem Red  Â£22.65       In stock   
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34       In stock   
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93       In stock   
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60       In stock   
9                                     The Black Maria  Â£52.15       In stock   
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99       In stock   
11    